# Context engineering trong agent

## Tổng quan

Phần khó nhất khi xây dựng agent (hoặc bất kỳ ứng dụng LLM nào) là làm cho chúng đủ tin cậy. Chúng có thể hoạt động tốt ở giai đoạn thử nghiệm, nhưng thường thất bại khi áp dụng vào các trường hợp thực tế.

### Vì sao agent thất bại?

Khi agent thất bại, nguyên nhân thường là do lệnh gọi LLM bên trong agent đã thực hiện sai hành động / không làm đúng như mong đợi. LLM thất bại vì một trong hai lý do:

1. LLM cơ bản không đủ năng lực
2. Context "đúng" đã không được truyền vào cho LLM

Trong đa số trường hợp — chính lý do thứ hai mới là nguyên nhân khiến agent hoạt động không ổn định.

**Context engineering** là việc cung cấp đúng thông tin và đúng công cụ theo đúng định dạng để LLM có thể hoàn thành một nhiệm vụ. Đây là công việc quan trọng số một của các AI Engineer. Việc thiếu context "đúng" chính là rào cản lớn nhất để xây dựng các agent đáng tin cậy hơn, và các abstraction (lớp trừu tượng) agent của LangChain được thiết kế đặc biệt để hỗ trợ context engineering.

<div class="alert alert-success">

Mới bắt đầu với context engineering? Hãy bắt đầu với [tổng quan khái niệm](https://docs.langchain.com/oss/python/concepts/context) để hiểu các loại context khác nhau và khi nào nên dùng chúng.

</div>

### Vòng lặp agent (agent loop)

Một vòng lặp agent thông thường bao gồm hai bước chính:

1. **Model call (lệnh gọi model)** - gọi LLM với một prompt và các tool có sẵn, trả về hoặc là một phản hồi hoặc là một yêu cầu thực thi tool
2. **Tool execution (thực thi tool)** - thực thi các tool mà LLM yêu cầu, trả về kết quả của tool

<p align="center">
    <img src="https://mintcdn.com/langchain-5e9cc07a/Tazq8zGc0yYUYrDl/oss/images/core_agent_loop.png?fit=max&auto=format&n=Tazq8zGc0yYUYrDl&q=85&s=ac72e48317a9ced68fd1be64e89ec063" height="300">
</p>

Vòng lặp này tiếp tục cho đến khi LLM quyết định kết thúc.

### Những gì bạn có thể kiểm soát

Để xây dựng các agent đáng tin cậy, bạn cần kiểm soát những gì xảy ra ở từng bước của vòng lặp agent, cũng như những gì xảy ra giữa các bước.

| Loại Context                                   | Bạn kiểm soát điều gì                                                                 | Tạm thời hay Bền vững |
| ----------------------------------------------- | -------------------------------------------------------------------------------------- | ---------------------- |
| **[Model Context](https://docs.langchain.com/oss/python/langchain/context-engineering#model-context)**             | Những gì được đưa vào lệnh gọi model (instruction, lịch sử tin nhắn, tool, định dạng phản hồi) | Tạm thời (Transient)   |
| **[Tool Context](https://docs.langchain.com/oss/python/langchain/context-engineering#tool-context)**               | Những gì tool có thể truy cập và tạo ra (đọc/ghi vào state, store, runtime context)     | Bền vững (Persistent)  |
| **[Life-cycle Context](https://docs.langchain.com/oss/python/langchain/context-engineering#life-cycle-context)**    | Những gì xảy ra giữa lệnh gọi model và tool (tóm tắt, guardrail, ghi log, v.v.)          | Bền vững (Persistent)  |

* **Context tạm thời**: Những gì LLM nhìn thấy trong một lệnh gọi đơn lẻ. Bạn có thể chỉnh sửa message, tool, hoặc prompt mà không làm thay đổi những gì được lưu trong state.
* **Context bền vững**: Những gì được lưu vào state qua các lượt tương tác. Các life-cycle hook và các lệnh ghi của tool sẽ thay đổi vĩnh viễn phần này.

### Các nguồn dữ liệu

Trong suốt quá trình này, agent của bạn truy cập (đọc / ghi) vào các nguồn dữ liệu khác nhau:

| Nguồn dữ liệu         | Còn được gọi là         | Phạm vi                | Ví dụ                                                                       |
| ---------------------- | ------------------------ | ----------------------- | ---------------------------------------------------------------------------- |
| **Runtime Context**     | Cấu hình tĩnh | Phạm vi trong một cuộc hội thoại | User ID, API key, kết nối cơ sở dữ liệu, quyền hạn, cấu hình môi trường      |
| **State**               | Bộ nhớ ngắn hạn          | Phạm vi trong một cuộc hội thoại | Các tin nhắn hiện tại, file đã tải lên, trạng thái xác thực, kết quả tool    |
| **Store**               | Bộ nhớ dài hạn           | Xuyên suốt nhiều cuộc hội thoại | Tùy chọn người dùng, thông tin chi tiết được trích xuất, ký ức, dữ liệu lịch sử |

### Cách hoạt động

[Middleware](https://docs.langchain.com/oss/python/langchain/middleware) của LangChain là cơ chế nền tảng giúp context engineering trở nên khả thi trong thực tế đối với các nhà phát triển sử dụng LangChain.

Middleware cho phép bạn "hook" (can thiệp) vào bất kỳ bước nào trong vòng đời của agent và:

* Cập nhật context
* Nhảy đến một bước khác trong vòng đời của agent

Trong suốt hướng dẫn này, bạn sẽ thấy middleware API được sử dụng thường xuyên như một phương tiện để thực hiện context engineering.

## Model context

Kiểm soát những gì được đưa vào mỗi lệnh gọi model - instruction, các tool có sẵn, model nào sẽ được sử dụng, và định dạng đầu ra. Những quyết định này ảnh hưởng trực tiếp đến độ tin cậy và chi phí.

* **System prompt**: Các chỉ dẫn cơ bản từ nhà phát triển gửi đến LLM.
* **Message**: Toàn bộ danh sách tin nhắn (lịch sử hội thoại) được gửi đến LLM.
* **Tool**: Các công cụ mà agent có thể sử dụng để thực hiện hành động.
* **Model**: Model thực tế (bao gồm cả cấu hình) sẽ được gọi.
* **Response format**: Đặc tả schema cho phản hồi cuối cùng của model.

Tất cả các loại model context này đều có thể lấy dữ liệu từ **state** (bộ nhớ ngắn hạn), **store** (bộ nhớ dài hạn), hoặc **runtime context** (cấu hình tĩnh).

### System prompt

System prompt quy định hành vi và năng lực của LLM. Những người dùng, ngữ cảnh, hoặc giai đoạn hội thoại khác nhau sẽ cần những chỉ dẫn khác nhau. Các agent thành công thường tận dụng ký ức, tùy chọn cá nhân và cấu hình để cung cấp đúng chỉ dẫn phù hợp với trạng thái hiện tại của cuộc hội thoại.

**State**
  
Truy cập số lượng tin nhắn hoặc ngữ cảnh hội thoại từ state:

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def state_aware_prompt(request: ModelRequest) -> str:
    # request.messages là một cách viết tắt của request.state["messages"]
    message_count = len(request.messages)

    base = "Bạn là một trợ lý hữu ích."

    if message_count > 10:
        base += "\nĐây là một cuộc hội thoại dài - hãy trả lời thật ngắn gọn."

    return base

agent = create_agent(
    model="gpt-5.5",
    tools=[...],
    middleware=[state_aware_prompt]
)

**Store**

Truy cập tùy chọn người dùng từ store:

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langgraph.store.memory import InMemoryStore

@dataclass
class Context:
    user_id: str

@dynamic_prompt
def store_aware_prompt(request: ModelRequest) -> str:
    user_id = request.runtime.context.user_id

    # Đọc từ Store: lấy tùy chọn của người dùng
    store = request.runtime.store
    user_prefs = store.get(("preferences",), user_id)

    base = "Bạn là một trợ lý hữu ích."

    if user_prefs:
        style = user_prefs.value.get("communication_style", "balanced")
        base += f"\nNgười dùng ưa thích phong cách trả lời {style}."

    return base

agent = create_agent(
    model="gpt-5.5",
    tools=[...],
    middleware=[store_aware_prompt],
    context_schema=Context,
    store=InMemoryStore()
)

**Runtime context**

Truy cập user ID hoặc cấu hình từ runtime context:

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class Context:
    user_role: str
    deployment_env: str

@dynamic_prompt
def context_aware_prompt(request: ModelRequest) -> str:
    # Đọc từ Runtime Context: vai trò người dùng và môi trường
    user_role = request.runtime.context.user_role
    env = request.runtime.context.deployment_env

    base = "Bạn là một trợ lý hữu ích."

    if user_role == "admin":
        base += "\nBạn có quyền admin. Bạn có thể thực hiện mọi thao tác."
    elif user_role == "viewer":
        base += "\nBạn chỉ có quyền xem. Hãy hướng dẫn người dùng thực hiện các thao tác đọc dữ liệu."

    if env == "production":
        base += "\nHãy đặc biệt cẩn trọng với mọi thao tác chỉnh sửa dữ liệu."

    return base

agent = create_agent(
    model="gpt-5.5",
    tools=[...],
    middleware=[context_aware_prompt],
    context_schema=Context
)

### Message

Message tạo thành prompt được gửi đến LLM. Việc quản lý nội dung của message là yếu tố then chốt để đảm bảo LLM có đủ thông tin đúng đắn để phản hồi tốt.

**State**

Chèn ngữ cảnh về file đã tải lên từ State khi liên quan đến câu hỏi hiện tại:

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def inject_file_context(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Chèn ngữ cảnh về các file mà người dùng đã tải lên trong phiên này."""
    # Đọc từ State: lấy metadata của các file đã tải lên
    uploaded_files = request.state.get("uploaded_files", [])

    if uploaded_files:
        # Xây dựng ngữ cảnh về các file hiện có
        file_descriptions = []
        for file in uploaded_files:
            file_descriptions.append(
                f"- {file['name']} ({file['type']}): {file['summary']}"
            )

        file_context = f"""Các file bạn có thể truy cập trong cuộc hội thoại này:
{chr(10).join(file_descriptions)}

Hãy tham chiếu các file này khi trả lời câu hỏi."""

        # Chèn ngữ cảnh file trước các tin nhắn gần nhất
        messages = [
            *request.messages,
            {"role": "user", "content": file_context},
        ]
        request = request.override(messages=messages)

    return handler(request)

agent = create_agent(
    model="gpt-5.5",
    tools=[...],
    middleware=[inject_file_context]
)

**Store**

Chèn phong cách viết email của người dùng từ Store để định hướng việc soạn thảo:

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
from langgraph.store.memory import InMemoryStore

@dataclass
class Context:
    user_id: str

@wrap_model_call
def inject_writing_style(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Chèn phong cách viết email của người dùng từ Store."""
    user_id = request.runtime.context.user_id

    # Đọc từ Store: lấy các ví dụ về phong cách viết của người dùng
    store = request.runtime.store
    writing_style = store.get(("writing_style",), user_id)

    if writing_style:
        style = writing_style.value
        # Xây dựng hướng dẫn phong cách từ các ví dụ đã lưu
        style_context = f"""Phong cách viết của bạn:
- Tông giọng: {style.get('tone', 'professional')}
- Lời chào thường dùng: "{style.get('greeting', 'Hi')}"
- Lời kết thường dùng: "{style.get('sign_off', 'Best')}"
- Email ví dụ bạn đã viết:
{style.get('example_email', '')}"""

        # Thêm vào cuối - model chú ý nhiều hơn đến các tin nhắn cuối cùng
        messages = [
            *request.messages,
            {"role": "user", "content": style_context}
        ]
        request = request.override(messages=messages)

    return handler(request)

agent = create_agent(
    model="gpt-5.5",
    tools=[...],
    middleware=[inject_writing_style],
    context_schema=Context,
    store=InMemoryStore()
)

**Runtime Context**

Chèn các quy định tuân thủ từ Runtime Context dựa trên khu vực pháp lý của người dùng:

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@dataclass
class Context:
    user_jurisdiction: str
    industry: str
    compliance_frameworks: list[str]

@wrap_model_call
def inject_compliance_rules(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Chèn các ràng buộc tuân thủ từ Runtime Context."""
    # Đọc từ Runtime Context: lấy các yêu cầu tuân thủ
    jurisdiction = request.runtime.context.user_jurisdiction
    industry = request.runtime.context.industry
    frameworks = request.runtime.context.compliance_frameworks

    # Xây dựng các ràng buộc tuân thủ
    rules = []
    if "GDPR" in frameworks:
        rules.append("- Phải có được sự đồng ý rõ ràng trước khi xử lý dữ liệu cá nhân")
        rules.append("- Người dùng có quyền yêu cầu xóa dữ liệu")
    if "HIPAA" in frameworks:
        rules.append("- Không được chia sẻ thông tin sức khỏe của bệnh nhân mà không có sự cho phép")
        rules.append("- Phải sử dụng phương thức giao tiếp an toàn, được mã hóa")
    if industry == "finance":
        rules.append("- Không được đưa ra lời khuyên tài chính nếu thiếu các tuyên bố miễn trừ trách nhiệm phù hợp")

    if rules:
        compliance_context = f"""Các yêu cầu tuân thủ đối với {jurisdiction}:
{chr(10).join(rules)}"""

        # Thêm vào cuối - model chú ý nhiều hơn đến các tin nhắn cuối cùng
        messages = [
            *request.messages,
            {"role": "user", "content": compliance_context}
        ]
        request = request.override(messages=messages)

    return handler(request)

agent = create_agent(
    model="gpt-5.5",
    tools=[...],
    middleware=[inject_compliance_rules],
    context_schema=Context
)

<div class="alert alert-info">

**Cập nhật tin nhắn tạm thời so với bền vững:**

Các ví dụ trên sử dụng `wrap_model_call` để thực hiện các cập nhật **tạm thời (transient)** - chỉnh sửa các tin nhắn được gửi đến model cho một lệnh gọi đơn lẻ mà không làm thay đổi những gì được lưu trong state.

Đối với các cập nhật **bền vững (persistent)** làm thay đổi state, bạn có thể:

* Trả về một [`ExtendedModelResponse`](https://reference.langchain.com/python/langchain/agents/middleware/types/ExtendedModelResponse) kèm theo một [`Command`](https://reference.langchain.com/python/langgraph/types/Command) từ `wrap_model_call` để chèn các cập nhật state ngay từ lớp model call.
* Sử dụng các life-cycle hook như `before_model`, `after_model`, hoặc `wrap_tool_call` (đối với kết quả trả về của tool) để cập nhật lịch sử hội thoại. Xem [tài liệu về middleware](https://docs.langchain.com/oss/python/langchain/middleware) để biết thêm chi tiết.

Xem [Cập nhật state](https://docs.langchain.com/oss/python/langchain/middleware/custom#state-updates) để biết thêm thông tin.

</div>

### Tool

Tool cho phép model tương tác với cơ sở dữ liệu, API, và các hệ thống bên ngoài. Cách bạn định nghĩa và lựa chọn tool sẽ ảnh hưởng trực tiếp đến việc model có thể hoàn thành nhiệm vụ một cách hiệu quả hay không.

#### Định nghĩa tool

Mỗi tool cần có một tên rõ ràng, mô tả, tên các tham số, và mô tả cho từng tham số. Đây không chỉ đơn thuần là metadata — chúng định hướng cách model suy luận về việc khi nào và cách nào để sử dụng tool.

In [ ]:
from langchain.tools import tool

@tool(parse_docstring=True)
def search_orders(
    user_id: str,
    status: str,
    limit: int = 10
) -> str:
    """Tìm kiếm đơn hàng của người dùng theo trạng thái.

    Sử dụng tool này khi người dùng hỏi về lịch sử đơn hàng hoặc muốn kiểm tra
    trạng thái đơn hàng. Luôn lọc theo trạng thái được cung cấp.

    Args:
        user_id: Định danh duy nhất của người dùng
        status: Trạng thái đơn hàng: 'pending' (đang chờ), 'shipped' (đã gửi), hoặc 'delivered' (đã giao)
        limit: Số lượng kết quả tối đa được trả về
    """
    # Triển khai logic tại đây
    pass

#### Lựa chọn tool

Không phải tool nào cũng phù hợp với mọi tình huống. Quá nhiều tool có thể gây quá tải cho model (làm ngập context) và làm tăng lỗi; quá ít tool sẽ hạn chế năng lực. Việc lựa chọn tool động sẽ điều chỉnh bộ tool có sẵn dựa trên trạng thái xác thực, quyền hạn người dùng, feature flag, hoặc giai đoạn của cuộc hội thoại.

**State**

Chỉ bật các tool nâng cao sau khi đạt đến một số mốc quan trọng trong hội thoại:

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def state_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Lọc tool dựa trên State của cuộc hội thoại."""
    # Đọc từ State: kiểm tra xem người dùng đã xác thực chưa
    state = request.state
    is_authenticated = state.get("authenticated", False)
    message_count = len(state["messages"])

    # Chỉ bật các tool nhạy cảm sau khi đã xác thực
    if not is_authenticated:
        tools = [t for t in request.tools if t.name.startswith("public_")]
        request = request.override(tools=tools)
    elif message_count < 5:
        # Hạn chế tool ở giai đoạn đầu của hội thoại
        tools = [t for t in request.tools if t.name != "advanced_search"]
        request = request.override(tools=tools)

    return handler(request)

agent = create_agent(
    model="gpt-5.5",
    tools=[public_search, private_search, advanced_search],
    middleware=[state_based_tools]
)

**Store**

Lọc tool dựa trên tùy chọn người dùng hoặc feature flag trong Store:

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
from langgraph.store.memory import InMemoryStore

@dataclass
class Context:
    user_id: str

@wrap_model_call
def store_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Lọc tool dựa trên tùy chọn trong Store."""
    user_id = request.runtime.context.user_id

    # Đọc từ Store: lấy các tính năng đã bật của người dùng
    store = request.runtime.store
    feature_flags = store.get(("features",), user_id)

    if feature_flags:
        enabled_features = feature_flags.value.get("enabled_tools", [])
        # Chỉ bao gồm các tool đã được bật cho người dùng này
        tools = [t for t in request.tools if t.name in enabled_features]
        request = request.override(tools=tools)

    return handler(request)

agent = create_agent(
    model="gpt-5.5",
    tools=[search_tool, analysis_tool, export_tool],
    middleware=[store_based_tools],
    context_schema=Context,
    store=InMemoryStore()
)

**Runtime Context**

Lọc tool dựa trên quyền hạn người dùng từ Runtime Context:

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@dataclass
class Context:
    user_role: str

@wrap_model_call
def context_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Lọc tool dựa trên quyền hạn từ Runtime Context."""
    # Đọc từ Runtime Context: lấy vai trò người dùng
    user_role = request.runtime.context.user_role

    if user_role == "admin":
        # Admin được cấp toàn bộ tool
        pass
    elif user_role == "editor":
        # Editor không được xóa
        tools = [t for t in request.tools if t.name != "delete_data"]
        request = request.override(tools=tools)
    else:
        # Viewer chỉ được cấp tool đọc dữ liệu
        tools = [t for t in request.tools if t.name.startswith("read_")]
        request = request.override(tools=tools)

    return handler(request)

agent = create_agent(
    model="gpt-5.5",
    tools=[read_data, write_data, delete_data],
    middleware=[context_based_tools],
    context_schema=Context
)

Xem [Tool động](https://docs.langchain.com/oss/python/langchain/tools#dynamic-tool-selection) để biết cách vừa lọc các tool đã đăng ký sẵn, vừa đăng ký tool tại thời điểm runtime (ví dụ: từ các MCP server).

### Model

Các model khác nhau có những điểm mạnh, chi phí và context window khác nhau. Hãy chọn đúng model cho từng nhiệm vụ cụ thể, và việc này có thể thay đổi trong suốt quá trình chạy của agent.

**State**

Sử dụng các model khác nhau dựa trên độ dài hội thoại từ State:

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

# Khởi tạo các model một lần bên ngoài middleware
large_model = init_chat_model("claude-sonnet-4-6")
standard_model = init_chat_model("gpt-5.5")
efficient_model = init_chat_model("gpt-5.4-mini")

@wrap_model_call
def state_based_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Chọn model dựa trên độ dài hội thoại trong State."""
    # request.messages là một cách viết tắt của request.state["messages"]
    message_count = len(request.messages)

    if message_count > 20:
        # Hội thoại dài - sử dụng model có context window lớn hơn
        model = large_model
    elif message_count > 10:
        # Hội thoại trung bình
        model = standard_model
    else:
        # Hội thoại ngắn - sử dụng model tiết kiệm chi phí
        model = efficient_model

    request = request.override(model=model)

    return handler(request)

agent = create_agent(
    model="gpt-5.4-mini",
    tools=[...],
    middleware=[state_based_model]
)

**Store**

Sử dụng model mà người dùng ưa thích từ Store:

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable
from langgraph.store.memory import InMemoryStore

@dataclass
class Context:
    user_id: str

# Khởi tạo các model có sẵn một lần
MODEL_MAP = {
    "gpt-5.5": init_chat_model("gpt-5.5"),
    "gpt-5.4-mini": init_chat_model("gpt-5.4-mini"),
    "claude-sonnet": init_chat_model("claude-sonnet-4-6"),
}

@wrap_model_call
def store_based_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Chọn model dựa trên tùy chọn trong Store."""
    user_id = request.runtime.context.user_id

    # Đọc từ Store: lấy model mà người dùng ưa thích
    store = request.runtime.store
    user_prefs = store.get(("preferences",), user_id)

    if user_prefs:
        preferred_model = user_prefs.value.get("preferred_model")
        if preferred_model and preferred_model in MODEL_MAP:
            request = request.override(model=MODEL_MAP[preferred_model])

    return handler(request)

agent = create_agent(
    model="gpt-5.5",
    tools=[...],
    middleware=[store_based_model],
    context_schema=Context,
    store=InMemoryStore()
)

**Runtime Context**

Chọn model dựa trên giới hạn chi phí hoặc môi trường từ Runtime Context:

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

@dataclass
class Context:
    cost_tier: str
    environment: str

# Khởi tạo các model một lần bên ngoài middleware
premium_model = init_chat_model("claude-sonnet-4-6")
standard_model = init_chat_model("gpt-5.5")
budget_model = init_chat_model("gpt-5.4-mini")

@wrap_model_call
def context_based_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Chọn model dựa trên Runtime Context."""
    # Đọc từ Runtime Context: mức chi phí và môi trường
    cost_tier = request.runtime.context.cost_tier
    environment = request.runtime.context.environment

    if environment == "production" and cost_tier == "premium":
        # Người dùng premium ở môi trường production được dùng model tốt nhất
        model = premium_model
    elif cost_tier == "budget":
        # Mức budget được dùng model tiết kiệm chi phí
        model = budget_model
    else:
        # Mức chuẩn (standard)
        model = standard_model

    request = request.override(model=model)

    return handler(request)

agent = create_agent(
    model="gpt-5.5",
    tools=[...],
    middleware=[context_based_model],
    context_schema=Context
)

Xem [Model động](https://docs.langchain.com/oss/python/langchain/models#dynamic-model-selection) để biết thêm các ví dụ.

### Response format

Structured output chuyển đổi văn bản không có cấu trúc thành dữ liệu có cấu trúc, đã được kiểm chứng. Khi cần trích xuất các trường dữ liệu cụ thể hoặc trả dữ liệu cho các hệ thống downstream, văn bản tự do là không đủ.

**Cách hoạt động:** Khi bạn cung cấp một schema làm response format, phản hồi cuối cùng của model được đảm bảo tuân theo schema đó. Agent sẽ chạy vòng lặp model / tool calling cho đến khi model không còn gọi tool nữa, sau đó phản hồi cuối cùng sẽ được ép buộc theo định dạng đã cung cấp.

#### Định nghĩa format

Các định nghĩa schema định hướng cho model. Tên trường, kiểu dữ liệu, và mô tả chỉ rõ chính xác định dạng mà đầu ra cần tuân theo.

In [ ]:
from pydantic import BaseModel, Field

class CustomerSupportTicket(BaseModel):
    """Thông tin ticket có cấu trúc được trích xuất từ tin nhắn của khách hàng."""

    category: str = Field(
        description="Danh mục vấn đề: 'billing' (thanh toán), 'technical' (kỹ thuật), 'account' (tài khoản), hoặc 'product' (sản phẩm)"
    )
    priority: str = Field(
        description="Mức độ khẩn cấp: 'low' (thấp), 'medium' (trung bình), 'high' (cao), hoặc 'critical' (nghiêm trọng)"
    )
    summary: str = Field(
        description="Tóm tắt vấn đề của khách hàng trong một câu"
    )
    customer_sentiment: str = Field(
        description="Trạng thái cảm xúc của khách hàng: 'frustrated' (bức xúc), 'neutral' (trung tính), hoặc 'satisfied' (hài lòng)"
    )

#### Lựa chọn format

Việc lựa chọn response format động sẽ điều chỉnh schema dựa trên tùy chọn người dùng, giai đoạn hội thoại, hoặc vai trò — trả về định dạng đơn giản ở giai đoạn đầu và định dạng chi tiết hơn khi độ phức tạp tăng lên.

**State**

Cấu hình structured output dựa trên state của hội thoại:

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from pydantic import BaseModel, Field
from typing import Callable

class SimpleResponse(BaseModel):
    """Phản hồi đơn giản cho giai đoạn đầu của hội thoại."""
    answer: str = Field(description="Một câu trả lời ngắn gọn")

class DetailedResponse(BaseModel):
    """Phản hồi chi tiết cho hội thoại đã diễn ra một thời gian."""
    answer: str = Field(description="Một câu trả lời chi tiết")
    reasoning: str = Field(description="Giải thích về quá trình suy luận")
    confidence: float = Field(description="Điểm tin cậy từ 0-1")

@wrap_model_call
def state_based_output(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Chọn định dạng đầu ra dựa trên State."""
    # request.messages là một cách viết tắt của request.state["messages"]
    message_count = len(request.messages)

    if message_count < 3:
        # Giai đoạn đầu hội thoại - sử dụng định dạng đơn giản
        request = request.override(response_format=SimpleResponse)
    else:
        # Hội thoại đã diễn ra một thời gian - sử dụng định dạng chi tiết
        request = request.override(response_format=DetailedResponse)

    return handler(request)

agent = create_agent(
    model="gpt-5.5",
    tools=[...],
    middleware=[state_based_output]
)

**Store**

Cấu hình định dạng đầu ra dựa trên tùy chọn người dùng trong Store:

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from pydantic import BaseModel, Field
from typing import Callable
from langgraph.store.memory import InMemoryStore

@dataclass
class Context:
    user_id: str

class VerboseResponse(BaseModel):
    """Phản hồi chi tiết, đầy đủ."""
    answer: str = Field(description="Câu trả lời chi tiết")
    sources: list[str] = Field(description="Các nguồn được sử dụng")

class ConciseResponse(BaseModel):
    """Phản hồi ngắn gọn."""
    answer: str = Field(description="Câu trả lời ngắn")

@wrap_model_call
def store_based_output(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Chọn định dạng đầu ra dựa trên tùy chọn trong Store."""
    user_id = request.runtime.context.user_id

    # Đọc từ Store: lấy phong cách phản hồi mà người dùng ưa thích
    store = request.runtime.store
    user_prefs = store.get(("preferences",), user_id)

    if user_prefs:
        style = user_prefs.value.get("response_style", "concise")
        if style == "verbose":
            request = request.override(response_format=VerboseResponse)
        else:
            request = request.override(response_format=ConciseResponse)

    return handler(request)

agent = create_agent(
    model="gpt-5.5",
    tools=[...],
    middleware=[store_based_output],
    context_schema=Context,
    store=InMemoryStore()
)

**Runtime Context**

Cấu hình định dạng đầu ra dựa trên Runtime Context như vai trò người dùng hoặc môi trường:

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from pydantic import BaseModel, Field
from typing import Callable

@dataclass
class Context:
    user_role: str
    environment: str

class AdminResponse(BaseModel):
    """Phản hồi kèm thông tin kỹ thuật chi tiết dành cho admin."""
    answer: str = Field(description="Câu trả lời")
    debug_info: dict = Field(description="Thông tin gỡ lỗi (debug)")
    system_status: str = Field(description="Trạng thái hệ thống")

class UserResponse(BaseModel):
    """Phản hồi đơn giản cho người dùng thông thường."""
    answer: str = Field(description="Câu trả lời")

@wrap_model_call
def context_based_output(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Chọn định dạng đầu ra dựa trên Runtime Context."""
    # Đọc từ Runtime Context: vai trò người dùng và môi trường
    user_role = request.runtime.context.user_role
    environment = request.runtime.context.environment

    if user_role == "admin" and environment == "production":
        # Admin ở môi trường production nhận được đầu ra chi tiết
        request = request.override(response_format=AdminResponse)
    else:
        # Người dùng thông thường nhận được đầu ra đơn giản
        request = request.override(response_format=UserResponse)

    return handler(request)

agent = create_agent(
    model="gpt-5.5",
    tools=[...],
    middleware=[context_based_output],
    context_schema=Context
)

## Tool context

Tool có đặc điểm riêng là chúng vừa đọc vừa ghi context.

Trong trường hợp cơ bản nhất, khi một tool được thực thi, nó nhận các tham số yêu cầu từ LLM và trả về một tool message. Tool sẽ thực hiện công việc của nó và tạo ra một kết quả.

Tool cũng có thể lấy các thông tin quan trọng cho model, giúp model có thể thực hiện và hoàn thành nhiệm vụ.

### Read

Đa số các tool trong thực tế cần nhiều hơn là chỉ các tham số từ LLM. Chúng cần user ID để truy vấn cơ sở dữ liệu, API key cho các dịch vụ bên ngoài, hoặc trạng thái session hiện tại để đưa ra quyết định. Tool đọc từ state, store, và runtime context để lấy các thông tin này.

**State**

Đọc từ State để kiểm tra thông tin session hiện tại:

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent

@tool
def check_authentication(
    runtime: ToolRuntime
) -> str:
    """Kiểm tra xem người dùng đã xác thực hay chưa."""
    # Đọc từ State: kiểm tra trạng thái xác thực hiện tại
    current_state = runtime.state
    is_authenticated = current_state.get("authenticated", False)

    if is_authenticated:
        return "Người dùng đã được xác thực"
    else:
        return "Người dùng chưa được xác thực"

agent = create_agent(
    model="gpt-5.5",
    tools=[check_authentication]
)

**Store**

Đọc từ Store để truy cập các tùy chọn người dùng đã được lưu:

In [ ]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore

@dataclass
class Context:
    user_id: str

@tool
def get_preference(
    preference_key: str,
    runtime: ToolRuntime[Context]
) -> str:
    """Lấy tùy chọn của người dùng từ Store."""
    user_id = runtime.context.user_id

    # Đọc từ Store: lấy các tùy chọn hiện có
    store = runtime.store
    existing_prefs = store.get(("preferences",), user_id)

    if existing_prefs:
        value = existing_prefs.value.get(preference_key)
        return f"{preference_key}: {value}" if value else f"Chưa có tùy chọn nào được đặt cho {preference_key}"
    else:
        return "Không tìm thấy tùy chọn nào"

agent = create_agent(
    model="gpt-5.5",
    tools=[get_preference],
    context_schema=Context,
    store=InMemoryStore()
)

**Runtime Context**

Đọc từ Runtime Context để lấy cấu hình như API key và user ID:

In [ ]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent

@dataclass
class Context:
    user_id: str
    api_key: str
    db_connection: str

@tool
def fetch_user_data(
    query: str,
    runtime: ToolRuntime[Context]
) -> str:
    """Lấy dữ liệu bằng cấu hình từ Runtime Context."""
    # Đọc từ Runtime Context: lấy API key và kết nối cơ sở dữ liệu
    user_id = runtime.context.user_id
    api_key = runtime.context.api_key
    db_connection = runtime.context.db_connection

    # Sử dụng cấu hình để lấy dữ liệu
    results = perform_database_query(db_connection, query, api_key)

    return f"Tìm thấy {len(results)} kết quả cho người dùng {user_id}"

agent = create_agent(
    model="gpt-5.5",
    tools=[fetch_user_data],
    context_schema=Context
)

# Gọi agent kèm runtime context
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Lấy dữ liệu của tôi"}]},
    context=Context(
        user_id="user_123",
        api_key="sk-...",
        db_connection="postgresql://..."
    )
)

### Writes (Ghi)

Kết quả của tool có thể được dùng để giúp agent hoàn thành một nhiệm vụ nhất định. Tool có thể vừa trả kết quả trực tiếp cho model
vừa cập nhật bộ nhớ của agent để cung cấp context quan trọng cho các bước tiếp theo.

<Tabs>
  <Tab title="State">
    Ghi vào State để theo dõi thông tin riêng của session bằng Command:

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    from langchain.tools import tool, ToolRuntime
    from langchain.agents import create_agent
    from langgraph.types import Command

    @tool
    def authenticate_user(
        password: str,
        runtime: ToolRuntime
    ) -> Command:
        """Xác thực người dùng và cập nhật State."""
        # Thực hiện xác thực (đã đơn giản hóa)
        if password == "correct":
            # Ghi vào State: đánh dấu là đã xác thực bằng Command
            return Command(
                update={"authenticated": True},
            )
        else:
            return Command(update={"authenticated": False})

    agent = create_agent(
        model="gpt-5.5",
        tools=[authenticate_user]
    )
```
  </Tab>

  <Tab title="Store">
    Ghi vào Store để lưu trữ dữ liệu xuyên suốt các session:

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    from dataclasses import dataclass
    from langchain.tools import tool, ToolRuntime
    from langchain.agents import create_agent
    from langgraph.store.memory import InMemoryStore

    @dataclass
    class Context:
        user_id: str

    @tool
    def save_preference(
        preference_key: str,
        preference_value: str,
        runtime: ToolRuntime[Context]
    ) -> str:
        """Lưu tùy chọn của người dùng vào Store."""
        user_id = runtime.context.user_id

        # Đọc các tùy chọn hiện có
        store = runtime.store
        existing_prefs = store.get(("preferences",), user_id)

        # Kết hợp với tùy chọn mới
        prefs = existing_prefs.value if existing_prefs else {}
        prefs[preference_key] = preference_value

        # Ghi vào Store: lưu các tùy chọn đã cập nhật
        store.put(("preferences",), user_id, prefs)

        return f"Đã lưu tùy chọn: {preference_key} = {preference_value}"

    agent = create_agent(
        model="gpt-5.5",
        tools=[save_preference],
        context_schema=Context,
        store=InMemoryStore()
    )
```
  </Tab>
</Tabs>

Xem [Tools](/oss/python/langchain/tools) để có các ví dụ đầy đủ về việc truy cập state, store, và runtime context trong tool.

## Life-cycle context

Kiểm soát những gì xảy ra **giữa** các bước cốt lõi của agent - can thiệp vào luồng dữ liệu để triển khai các mối quan tâm xuyên suốt (cross-cutting concerns) như tóm tắt, guardrail, và ghi log.

Như bạn đã thấy trong phần [Model Context](#model-context) và [Tool Context](#tool-context), [middleware](/oss/python/langchain/middleware) là cơ chế giúp context engineering trở nên khả thi trong thực tế. Middleware cho phép bạn "hook" vào bất kỳ bước nào trong vòng đời của agent và:

1. **Cập nhật context** - Chỉnh sửa state và store để lưu lại các thay đổi, cập nhật lịch sử hội thoại, hoặc lưu lại các thông tin chi tiết
2. **Nhảy trong vòng đời** - Chuyển đến các bước khác trong chu kỳ của agent dựa trên context (ví dụ: bỏ qua việc thực thi tool nếu một điều kiện nào đó được đáp ứng, lặp lại lệnh gọi model với context đã được chỉnh sửa)

<div style={{ display: "flex", justifyContent: "center" }}>
  <img src="https://mintcdn.com/langchain-5e9cc07a/RAP6mjwE5G00xYsA/oss/images/middleware_final.png?fit=max&auto=format&n=RAP6mjwE5G00xYsA&q=85&s=eb4404b137edec6f6f0c8ccb8323eaf1" alt="Các hook của middleware trong vòng lặp agent" className="rounded-lg" width="500" height="560" data-path="oss/images/middleware_final.png" />
</div>

### Ví dụ: Tóm tắt (Summarization)

Một trong những pattern (mẫu thiết kế) life-cycle phổ biến nhất là tự động rút gọn lịch sử hội thoại khi nó trở nên quá dài. Khác với việc cắt bớt tin nhắn tạm thời (transient message trimming) đã được trình bày trong phần [Model Context](#messages), việc tóm tắt sẽ **cập nhật state một cách bền vững (persistently)** - thay thế vĩnh viễn các tin nhắn cũ bằng một bản tóm tắt được lưu lại cho tất cả các lượt tương tác sau này.

LangChain cung cấp sẵn middleware cho việc này:

```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5.5",
    tools=[...],
    middleware=[
        SummarizationMiddleware(
            model="gpt-5.4-mini",
            trigger={"tokens": 4000},
            keep=("messages", 20),
        ),
    ],
)
```

Khi cuộc hội thoại vượt quá giới hạn token, `SummarizationMiddleware` sẽ tự động:

1. Tóm tắt các tin nhắn cũ hơn bằng một lệnh gọi LLM riêng biệt
2. Thay thế chúng bằng một tin nhắn tóm tắt trong State (vĩnh viễn)
3. Giữ nguyên các tin nhắn gần đây để duy trì context

Lịch sử hội thoại đã được tóm tắt sẽ được cập nhật vĩnh viễn - các lượt tương tác sau này sẽ nhìn thấy bản tóm tắt thay vì các tin nhắn ban đầu.

<Note>
  Để biết danh sách đầy đủ các middleware có sẵn, các hook khả dụng, và cách tạo middleware tùy chỉnh, xem [tài liệu về Middleware](/oss/python/langchain/middleware).
</Note>

## Các phương pháp thực hành tốt nhất (Best practices)

1. **Bắt đầu đơn giản** - Bắt đầu với các prompt và tool tĩnh, chỉ thêm các yếu tố động khi thực sự cần thiết
2. **Kiểm thử theo từng bước** - Chỉ thêm từng tính năng context engineering một lúc
3. **Theo dõi hiệu suất** - Theo dõi các lệnh gọi model, mức sử dụng token, và độ trễ (latency)
4. **Sử dụng middleware có sẵn** - Tận dụng [`SummarizationMiddleware`](/oss/python/langchain/middleware#summarization), [`LLMToolSelectorMiddleware`](/oss/python/langchain/middleware#llm-tool-selector), v.v.
5. **Ghi lại chiến lược context của bạn** - Làm rõ context nào đang được truyền và tại sao
6. **Hiểu rõ sự khác biệt giữa tạm thời và bền vững**: Các thay đổi ở model context là tạm thời (theo từng lệnh gọi), trong khi các thay đổi ở life-cycle context sẽ được lưu lại vào state một cách bền vững

## Tài nguyên liên quan

* [Tổng quan khái niệm về Context](/oss/python/concepts/context) - Hiểu các loại context và khi nào nên sử dụng chúng
* [Middleware](/oss/python/langchain/middleware) - Hướng dẫn đầy đủ về middleware
* [Tools](/oss/python/langchain/tools) - Tạo tool và truy cập context
* [Memory (Bộ nhớ)](/oss/python/concepts/memory) - Các pattern bộ nhớ ngắn hạn và dài hạn
* [Agents](/oss/python/langchain/agents) - Các khái niệm cốt lõi về agent

***

<div className="source-links">
  <Callout icon="terminal-2">
    [Kết nối tài liệu này](/use-these-docs) với Claude, VSCode, và nhiều công cụ khác qua MCP để nhận câu trả lời theo thời gian thực.
  </Callout>

  <Callout icon="edit">
    [Chỉnh sửa trang này trên GitHub](https://github.com/langchain-ai/docs/edit/main/src/oss/langchain/context-engineering.mdx) hoặc [báo lỗi (file an issue)](https://github.com/langchain-ai/docs/issues/new/choose).
  </Callout>
</div>